<a href="https://colab.research.google.com/github/anshuman-dataverse/Gen_ai/blob/main/M5_Lab2_LangChain_Templates_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- applied-genai-header -->
<div style="background: linear-gradient(135deg, #1e4b8f 0%, #2d6cb8 100%); color: white; padding: 24px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; margin-bottom: 16px;">
  <div style="font-size: 12px; opacity: 0.85; letter-spacing: 1.5px; text-transform: uppercase;">Applied Generative AI · IE 5373</div>
  <h1 style="margin: 8px 0 4px 0; font-size: 28px; font-weight: 600;">M5 Lab 2 — Prompt Templates & Memory</h1>
  <div style="font-size: 14px; opacity: 0.9;">PromptTemplate, ConversationBuffer / Window / Summary memory</div>
  <div style="margin-top: 12px; font-size: 12px; opacity: 0.8;">Prof. Mohammad Dehghani · Northeastern University</div>
</div>

> **📌 Note on models.** This lab references specific LLM versions (e.g. `gpt-5`, `gpt-5-mini`).
> Models update quickly — you are welcome (and encouraged) to swap in any newer OpenAI / Anthropic / Google model you have access to.
> The default model is set in one place: `DEFAULT_CHAT_MODEL` inside `utils.py`. Change it there and every cell follows.


In [ ]:
# === Shared lab setup: utils.py + API key + sticky lab pill ===
# Downloads the shared utilities (pretty_print, model constants, key loader,
# lab_pill) from the AppliedGenAI repo so every notebook stays small and
# consistent. The API key is read from a Colab secret named OPENAI_API_KEY
# (set it once under Colab → 🔑 → "Notebook access" — same name in every lab).
import os
if not os.path.exists("utils.py"):
    !wget -q https://raw.githubusercontent.com/mdehghani86/AppliedGenAI/main/utils.py -O utils.py

from utils import (
    pretty_print,
    DEFAULT_CHAT_MODEL,   # e.g. "gpt-5"  — main reasoning model
    DEFAULT_MINI_MODEL,   # e.g. "gpt-5-mini"  — cheaper / faster default
    DEFAULT_EMBED_MODEL,  # e.g. "text-embedding-3-small"
    get_openai_key,
    lab_pill,
)

lab_pill('M5 Lab 2 — Prompt Templates & Memory')        # sticky banner so you always see which lab you're in
get_openai_key(verify=True)    # loads the key + pings OpenAI to confirm it works


<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 26px 28px 18px 28px; border-radius: 14px; margin-bottom: 22px; font-family: Arial, sans-serif;">
  <h2 style="margin-top: 0; font-size: 27px; letter-spacing: 0.5px;">
    🧠 LangChain Lab 2: Prompt Templates &amp; Memory
  </h2>
  <p style="font-size: 17px; margin-bottom: 8px;">
    <span style="color: #a5d8ff;">Instructor:</span> Prof. Dehghani
  </p>
  <h3 style="color: #a5d8ff; font-size: 19px; margin-bottom: 12px;">Lab Overview</h3>
  <p style="font-size: 16px; margin-bottom: 14px;">
    In this lab, you'll enhance your interactions with LLMs by using <b>Prompt Templates</b> and <b>Memory</b> features in LangChain.<br>
    You'll learn to create <b>structured prompts dynamically</b> and maintain <b>conversation history</b> across multiple turns.
  </p>
  <hr style="border: 1px solid #3f77d4; margin: 16px 0;">
  <h3 style="color: #a5d8ff; font-size: 19px; margin-bottom: 10px;">🎯 What You'll Learn</h3>
  <ul style="font-size: 16px; margin: 0 0 10px 22px; line-height: 1.8;">
    <li>🔹 <b>Prompt Templates</b> – Format inputs dynamically for LLMs.</li>
    <li>🔹 <b>Memory in LangChain</b> – Maintain context in multi-turn conversations.</li>
    <li>🔹 <b>Hands-on exercises</b> – Reinforce concepts with practical coding tasks.</li>
  </ul>
  <p style="font-size: 15.5px; margin-top: 8px;">
    By the end, you'll be able to structure prompts effectively and implement conversational memory in LangChain applications. 🚀
  </p>
</div>


##⚙️ Install essential packages

In [2]:
#⚙️ Install essential packages for LangChain with OpenAI & Gemini support

!pip -q install -U \
  langchain \
  langchain-core \
  langchain-community \
  langchain-openai \
  langchain-google-genai \
  langchain-classic \
  openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

##🔑 Step 2: Set Up OpenAI API Key

In [9]:
# ⚙️ Load API Keys from Colab Secrets
# ==================================

import os                                  # Used to set environment variables for API keys
from google.colab import userdata          # To securely access stored secrets in Colab

# Retrieve your stored secrets (API keys)
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')   # OpenAI API key for GPT models
GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')   # Google Gemini API key for Gemini models

# Set environment variables for the APIs and confirm success
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # Set OpenAI key as environment variable
    print("✅ OpenAI API key loaded successfully!")
else:
    print("❌ OpenAI API key not found. Please set 'OPENAI_API_KEY' in Colab secrets.")

if GEMINI_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY   # Set Gemini key as environment variable
    print("✅ Google Gemini API key loaded successfully!")
else:
    print("❌ Google Gemini API key not found. Please set 'GEMINI_API_KEY' in Colab secrets.")

✅ OpenAI API key loaded successfully!
✅ Google Gemini API key loaded successfully!


<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 26px 28px 18px 28px; border-radius: 14px; margin-bottom: 22px; font-family: Arial, sans-serif;">
  <h2 style="margin-top: 0; font-size: 25px; letter-spacing: 0.5px;">📝 Prompt Templates in LangChain</h2>

  <h3 style="color: #a5d8ff; font-size: 19px; margin-bottom: 8px;">🔹 What are Prompt Templates?</h3>
  <p style="font-size: 16px; margin-bottom: 12px;">
    Prompt Templates let you <b>dynamically format prompts</b> by inserting variables, making interactions with LLMs more flexible and reusable.<br>
    Instead of writing static text, you can use placeholders that are filled in with real values at runtime.
  </p>

  <h3 style="color: #a5d8ff; font-size: 19px; margin-bottom: 8px;">🔹 Why Use Prompt Templates?</h3>
  <ul style="font-size: 16px; margin: 0 0 14px 22px; line-height: 1.8;">
    <li>✅ <b>Reusability</b> – No need for repetitive prompts.</li>
    <li>✅ <b>Dynamic Inputs</b> – Easily personalize prompts with new user data.</li>
    <li>✅ <b>Consistency</b> – Keeps your prompt formatting structured and reliable.</li>
  </ul>

  <h3 style="font-size: 17px; margin-bottom: 8px;">📌 Example Usage</h3>
  <div style="background: rgba(255,255,255,0.08); border-radius: 7px; padding: 11px 16px; margin-bottom: 6px;">
    <span style="color: #a5d8ff;">Static prompt:</span><br>
    <span style="font-family: 'Fira Mono', monospace; font-size: 15px; color: #fff;">
      "What are the benefits of AI in healthcare?"
    </span>
  </div>
  <div style="background: rgba(255,255,255,0.08); border-radius: 7px; padding: 11px 16px;">
    <span style="color: #a5d8ff;">Dynamic prompt with a template:</span><br>
    <span style="font-family: 'Fira Mono', monospace; font-size: 15px; color: #fff;">
      "What are the benefits of {technology} in {industry}?"
    </span><br>
    <span style="font-size: 15px;">If <b>{technology} = "AI"</b> and <b>{industry} = "healthcare"</b>, the prompt becomes:</span><br>
    <span style="font-family: 'Fira Mono', monospace; font-size: 15px; color: #fff;">
      "What are the benefits of AI in healthcare?"
    </span>
  </div>

  <p style="font-size: 16px; margin-top: 14px;">🚀 Let's get started with the first example!</p>
</div>


In [10]:
"""
# ==================================================
# 🎯 Using Prompt Templates with OpenAI (GPT-4)
# ==================================================
"""
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI   # ✅ NEW import for ChatOpenAI

# Step 1: Define a prompt template with variables
prompt_template = PromptTemplate(
    input_variables=["technology", "industry"],
    template="What are the benefits of {technology} in {industry} in 1 sentence?"
)

# Step 2: Format the prompt with specific values
formatted_prompt = prompt_template.format(technology="Drones", industry="SupplyChain")

# Step 3: Initialize the OpenAI LLM (GPT-4)
llm_ChatGPT = ChatOpenAI(model=DEFAULT_CHAT_MODEL)

# Step 4: Generate the response
response_ChatGPT = llm_ChatGPT.invoke(formatted_prompt)

# Step 5: Display results
print("🔹 Generated Prompt:", formatted_prompt)
pretty_print(response_ChatGPT.content, title="🔹 LLM Response:")


🔹 Generated Prompt: What are the benefits of Drones in SupplyChain in 1 sentence?


In [11]:
# ==================================================
# ✋ **Hands-On: Creating Dynamic Prompt Templates** — COMPLETED
# ==================================================

# 📌 Task: Fill in the missing placeholders so the PromptTemplate
# correctly substitutes {topic} and {context}, then send the formatted
# prompt to GPT-4 and print the response.

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# ✅ Step 1: Define a Prompt Template
prompt_template = PromptTemplate(
    input_variables=["topic", "context"],  # both placeholder names listed
    template="How does {topic} impact {context} in a few words?"
)

# ✅ Step 2: Format the prompt with actual values
formatted_prompt = prompt_template.format(
    topic="Machine Learning",
    context="business analytics"
)

# ✅ Step 3: Generate a response using OpenAI (GPT-4)
llm_ChatGPT = ChatOpenAI(model=DEFAULT_CHAT_MODEL)   # instantiate the chat model
response_ChatGPT = llm_ChatGPT.invoke(formatted_prompt)

# ✅ Step 4: Display results
print("🔹 **Generated Prompt:**", formatted_prompt)
print("🔹 **LLM Response:**", response_ChatGPT.content)


🔹 **Generated Prompt:** How does Machine Learning impact business analytics in a few words?
🔹 **LLM Response:** Enhances decision-making, automates data analysis, uncovers insights, and improves predictive accuracy.


In [12]:
"""
# ==================================================
# 🔄 Using Prompt Templates in a Loop
# ==================================================
"""

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Step 1️⃣: Define a prompt template with variables
prompt_template = PromptTemplate(
    input_variables=["technology", "industry"],
    template="In one sentence, how does {technology} impact {industry} in 1 sentence?"
)

# Step 2️⃣: Initialize the OpenAI LLM (GPT-4)
llm_ChatGPT = ChatOpenAI(model=DEFAULT_CHAT_MODEL)

# Step 3️⃣: Define input values for the loop
input_data = [
    {"technology": "AI", "industry": "education"},
    {"technology": "Blockchain", "industry": "finance"},
    {"technology": "5G", "industry": "telecommunications"},
]

# Step 4️⃣: Loop through inputs, format the prompt, and generate a response
for data in input_data:
    formatted_prompt = prompt_template.format(**data)
    response_ChatGPT = llm_ChatGPT.invoke(formatted_prompt)

    # Step 5️⃣: Display results in a clear, modern format
    print(f"🔹 Prompt: {formatted_prompt}")
    pretty_print(response_ChatGPT.content, title="💡 Response")
    print("-" * 60)

🔹 Prompt: In one sentence, how does AI impact education in 1 sentence?


------------------------------------------------------------
🔹 Prompt: In one sentence, how does Blockchain impact finance in 1 sentence?


------------------------------------------------------------
🔹 Prompt: In one sentence, how does 5G impact telecommunications in 1 sentence?


------------------------------------------------------------


<div style="background: linear-gradient(135deg, #1a386e 0%, #377ce8 100%); color: white; padding: 24px 26px 16px 26px; border-radius: 12px; margin-bottom: 22px; font-family: Arial, sans-serif;">
  <h3 style="margin-top: 0; font-size: 21px;">🔗 Wrapping Up: Why Prompt Templates Matter</h3>
  <p style="font-size: 16px;">
    Just as you wouldn't rewrite a whole menu for every customer in a coffee shop, you don't need to create a new prompt for every question you ask an LLM.
    <br><br>
    <b>Prompt templates</b> give you a reusable, flexible, and structured way to interact with language models—making your code cleaner, your queries more consistent, and your applications easier to scale.
    <br><br>
    Whether you're building a chatbot, automating business tasks, or analyzing data, prompt templates are an essential tool in your GenAI toolkit!
  </p>
</div>


<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 26px 28px 18px 28px; border-radius: 14px; margin-bottom: 22px; font-family: Arial, sans-serif;">
  <h2 style="margin-top: 0; font-size: 27px; letter-spacing: 0.5px;">
    🧠 Understanding Memory in LangChain
  </h2>

  <h3 style="color: #a5d8ff; font-size: 19px; margin-bottom: 10px;">🔹 What is Memory in LangChain?</h3>
  <p style="font-size: 16px; margin-bottom: 13px;">
    By default, LLMs <b>do not remember past interactions</b>.<br>
    LangChain <b>Memory</b> allows an AI model to <b>retain context</b> across multiple turns, enabling more natural, conversational interactions.
  </p>

  <h3 style="color: #a5d8ff; font-size: 19px; margin-bottom: 10px;">🔹 Why Use Memory?</h3>
  <ul style="font-size: 16px; margin: 0 0 14px 22px; line-height: 1.8;">
    <li>✅ <b>Maintains conversation history</b> – AI can recall previous exchanges.</li>
    <li>✅ <b>Improves response coherence</b> – Reduces redundant user re-explanations.</li>
    <li>✅ <b>Essential for chatbots &amp; agents</b> – Allows multi-turn dialogue without loss of context.</li>
  </ul>

  <h3 style="color: #a5d8ff; font-size: 19px; margin-bottom: 10px;">🔹 Types of Memory in LangChain</h3>
  <ul style="font-size: 16px; margin: 0 0 14px 22px; line-height: 1.7;">
    <li>1️⃣ <b>ConversationBufferMemory</b> – Stores messages in a buffer (basic memory).</li>
    <li>2️⃣ <b>ConversationSummaryMemory</b> – Summarizes past interactions instead of storing all messages.</li>
    <li>3️⃣ <b>ConversationBufferWindowMemory</b> – Retains only the last N interactions for efficiency.</li>
    <li>4️⃣ <b>Vector-based Memory</b> – Uses embeddings for advanced retrieval of past conversations.</li>
  </ul>

  <h3 style="font-size: 17px; margin-bottom: 8px;">🚀 What We'll Do in This Lab</h3>
  <p style="font-size: 16px;">
    We’ll start with <b>ConversationBufferMemory</b>, which allows an LLM to <b>recall past messages</b> and interact in a more natural, memory-enhanced way.<br>
    <br>
    <span style="color: #a5d8ff;">Note:</span> When using a conversation chain with memory, you’ll use the <b><code>predict()</code></b> method instead of <code>invoke()</code>. This lets the AI maintain and use context across multiple turns.<br>
    <br>
    Let's get started! 👇
  </p>
</div>


In [13]:
"""
# ==================================================
# 💬🧠 Memory Matters: AnniversaryBot Demo (Stateless vs. Memory)
# ==================================================
#
# This demo shows how LangChain's memory feature allows an AI assistant to remember
# details—using the example of a user telling the bot their anniversary date.
"""

from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

# -------------------------------
# 1️⃣ Version WITHOUT Memory (Stateless)
# -------------------------------
llm_stateless = ChatOpenAI(model=DEFAULT_CHAT_MODEL)

print("\n========== WITHOUT MEMORY (Stateless LLM) ==========")
print("👩‍❤️‍👨 User: Our anniversary is October 5th. Please remember that!")
response1 = llm_stateless.invoke("Our anniversary is October 5th. Please remember that!")
pretty_print(response1.content, title="🤖 AnniversaryBot:")

print("👩‍❤️‍👨 User: When is our anniversary?")
response2 = llm_stateless.invoke("When is our anniversary?")
pretty_print(response2.content, title="🤖 AnniversaryBot:")

# -------------------------------
# 2️⃣ Version WITH Memory
# -------------------------------
memory = ConversationBufferMemory()
llm_with_mem = ChatOpenAI(model=DEFAULT_CHAT_MODEL)
conversation_with_mem = ConversationChain(llm=llm_with_mem, memory=memory)

print("\n========== WITH MEMORY ==========")
print("👩‍❤️‍👨 User: Our anniversary is October 5th. Please remember that!")
response3 = conversation_with_mem.predict(input="Our anniversary is October 5th. Please remember that!")
print("🤖 AnniversaryBot:", response3)

print("👩‍❤️‍👨 User: When is our anniversary?")
response4 = conversation_with_mem.predict(input="When is our anniversary?")
print("🤖 AnniversaryBot:", response4)



========== WITHOUT MEMORY (Stateless LLM) ==========
👩‍❤️‍👨 User: Our anniversary is October 5th. Please remember that!


👩‍❤️‍👨 User: When is our anniversary?


/tmp/ipykernel_1151/167030562.py:31: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory()
/tmp/ipykernel_1151/167030562.py:33: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation_with_mem = ConversationChain(llm=llm_with_mem, memory=memory)



========== WITH MEMORY ==========
👩‍❤️‍👨 User: Our anniversary is October 5th. Please remember that!
🤖 AnniversaryBot: Congratulations on your anniversary! October 5th is a wonderful time of year to celebrate. Did you know that in 1582 the Gregorian calendar was introduced in October by Pope Gregory XIII to replace the Julian calendar? It's amazing how different aspects of time and dates weave through history. Do you have any special plans for your celebration?
👩‍❤️‍👨 User: When is our anniversary?
🤖 AnniversaryBot: Your anniversary is on October 5th. That's a special date to celebrate! If you're looking for ideas to make the day memorable, consider planning a cozy dinner at home, taking a weekend getaway to a favorite location, or even crafting a personalized gift that reflects your journey together. Whatever you choose, I hope it's a wonderful celebration!


In [15]:
# ==================================================
# ✋ Hands-On: Using Memory with OpenAI (Beer Game — Supply Chain) — COMPLETED
# ==================================================

# 📌 Task: Build a ConversationChain with ConversationBufferMemory so that
# the model remembers what the retailer said in earlier turns.

from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

# ✅ Step 1: Initialize Memory
memory = ConversationBufferMemory()   # stores the full transcript turn-by-turn

# ✅ Step 2: Initialize ChatGPT with Memory
llm_ChatGPT = ChatOpenAI(model=DEFAULT_CHAT_MODEL)
conversation = ConversationChain(llm=llm_ChatGPT, memory=memory)

# ✅ Step 3: Run Multiple Interactions
print("\n💬 **Retailer:** Last week, the customer demand was 200 units. What should I order this week?")
response_ChatGPT = conversation.predict(
    input="Last week, the customer demand was 200 units. What should I order this week?"
)
print("🤖 **ChatGPT:**", response_ChatGPT)

print("\n💬 **Retailer:** If demand increases by 10%, how many units should I prepare for next week?")
response_ChatGPT = conversation.predict(
    input="If demand increases by 10%, how many units should I prepare for next week?"
)
print("🤖 **ChatGPT:**", response_ChatGPT)

print("\n💬 **Retailer:** What was the demand I mentioned last week?")
response_ChatGPT = conversation.predict(
    input="What was the demand I mentioned last week?"
)
print("🤖 **ChatGPT:**", response_ChatGPT)

# ✅ Inspect what is sitting in memory
print("\n📜 **Stored memory after all turns:**")
print(memory.load_memory_variables({})["history"])



💬 **Retailer:** Last week, the customer demand was 200 units. What should I order this week?
🤖 **ChatGPT:** When determining how much to order this week, you'll want to consider several factors beyond just last week's demand of 200 units. Here are a few things to consider:

1. **Trends and Patterns**: Have you observed any demand patterns or trends over time? Are sales typically higher during certain weeks or months?

2. **Lead Time**: How long does it take for the new stock to arrive once you've placed an order? This can affect how much you need to order to ensure there's no stockout.

3. **Safety Stock**: It's often a good idea to maintain some safety stock in case of unexpected spikes in demand or delays in the supply chain.

4. **Future Demand Forecasts**: Are there upcoming events, promotions, or seasonal factors that might affect demand in the near future?

5. **Current Inventory Levels**: How much inventory do you currently have on hand? You’ll want to ensure this inventory, pl

## Using 'Summarized Conversation' Example


In [16]:
# ==================================================
# 🎤 **Using Memory in LangChain: Job Interview Prep**
# ==================================================
#
# This script simulates a job interview practice session.
# It uses ConversationSummaryMemory to retain key points from previous exchanges
# rather than storing the full conversation history.

# ✅ Import required classes
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory  # Summarized conversation memory
from langchain_classic.chains import ConversationChain

# ✅ Step 1: Initialize Memory
# This memory will maintain a **summarized** version of the conversation.
memory = ConversationSummaryMemory(llm=ChatOpenAI(model=DEFAULT_CHAT_MODEL))

# ✅ Step 2: Initialize ChatGPT with Memory
llm = ChatOpenAI(model=DEFAULT_CHAT_MODEL)  # Using GPT-4 model

# ✅ Step 3: Initialize Conversation Chain
# The model will summarize key details from the job interview practice.
conversation = ConversationChain(llm=llm, memory=memory)

# ✅ Step 4: Conduct the Interview Simulation

print("\n💬 **User:** Can you ask me a common interview question?")
response = conversation.predict(input="Can you ask me a common interview question?")
print("🤖 **ChatGPT:**", response)

# ✅ Check memory after first interaction
print("\n📜 **Memory Summary After 1st Question:**")
print(memory.load_memory_variables({})["history"])

print("\n💬 **User:** My biggest strength is adaptability and problem-solving.")
response = conversation.predict(input="My biggest strength is adaptability and problem-solving.")
print("🤖 **ChatGPT:**", response)

# ✅ Check memory after user shares strength
print("\n📜 **Memory Summary After Strength Response:**")
print(memory.load_memory_variables({})["history"])

print("\n💬 **User:** My biggest weakness is that I sometimes overthink decisions.")
response = conversation.predict(input="My biggest weakness is that I sometimes overthink decisions.")
print("🤖 **ChatGPT:**", response)

# ✅ Check memory after user shares weakness
print("\n📜 **Memory Summary After Weakness Response:**")
print(memory.load_memory_variables({})["history"])

print("\n💬 **User:** Can you summarize what we discussed so far?")
response = conversation.predict(input="Can you summarize what we discussed so far?")
print("🤖 **ChatGPT:**", response)

# ✅ Final Memory Check
print("\n📜 **Final Memory Summary:**")
print(memory.load_memory_variables({})["history"])



💬 **User:** Can you ask me a common interview question?
🤖 **ChatGPT:** Sure! A common interview question is: "Can you tell me about a time when you faced a challenge at work and how you handled it?" Interviewers use this question to gauge your problem-solving skills, resilience, and ability to handle pressure. When answering, you might want to use the STAR method (Situation, Task, Action, Result) to structure your response effectively. Would you like help crafting a response using this method?

📜 **Memory Summary After 1st Question:**
The AI provides a common interview question, "Can you tell me about a time when you faced a challenge at work and how you handled it?" It explains that interviewers use this question to assess problem-solving skills, resilience, and handling pressure. The AI also suggests using the STAR method (Situation, Task, Action, Result) to structure responses and offers help in crafting a response.

💬 **User:** My biggest strength is adaptability and problem-solvi

In [17]:
# ==================================================
# 🍺 **LangChain Beer Game: Comparing Memory Types
# ==================================================
#
# This script simulates a Beer Game ordering process over 6 weeks.
# It uses:
# 1️⃣ ConversationBufferMemory (Tracks full history)
# 2️⃣ ConversationBufferWindowMemory (Tracks only last 3 orders)
#
# The AI predicts the next order quantity based on past interactions.

# ✅ Import required libraries
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory
from langchain_core.prompts import PromptTemplate

# ✅ Step 1: Initialize Memory Types
buffer_memory = ConversationBufferMemory(return_messages=True)  # Stores entire history
window_memory = ConversationBufferWindowMemory(k=3, return_messages=True)  # Stores last 3 interactions

# ✅ Step 2: Initialize Chat Model (NEW!)
llm = ChatOpenAI(model=DEFAULT_CHAT_MODEL)

# ✅ Step 3: Define a Prompt Template
beer_game_template = PromptTemplate(
    input_variables=["context"],
    template="""
    You are managing a supply chain for a beer distribution system.
    Orders fluctuate at first but stabilize later.

    {context}

    Based on past orders, what should be the next order quantity?
    """
)


# ✅ Step 4: Define Processing Pipelines
buffer_chain = beer_game_template | llm
window_chain = beer_game_template | llm

# ✅ Step 5: Define Order Fluctuations (First 3 weeks volatile, last 3 weeks stable)
weekly_orders = [20, 50, 10, 25, 30, 30]  # Example fluctuations

# Store results for comparison
buffer_memory_log = []
window_memory_log = []
buffer_predictions = []
window_predictions = []

# ✅ Step 6: Run the Simulation
for week in range(1, len(weekly_orders) + 1):
    prev_orders = ", ".join(map(str, weekly_orders[:week]))  # Orders seen so far
    context = f"Week {week}: The previous orders were {prev_orders}."

    # Store input in memory
    buffer_memory.save_context({"context": context}, {"response": ""})
    window_memory.save_context({"context": context}, {"response": ""})

    # Get AI predictions using RunnableSequence
    buffer_prediction = buffer_chain.invoke({"context": context})
    window_prediction = window_chain.invoke({"context": context})

    # Retrieve memory states
    buffer_memory_summary = buffer_memory.load_memory_variables({})["history"]
    window_memory_summary = window_memory.load_memory_variables({})["history"]

    # Store memory states and predictions
    buffer_memory_log.append(buffer_memory_summary)
    window_memory_log.append(window_memory_summary)
    buffer_predictions.append(buffer_prediction.content)
    window_predictions.append(window_prediction.content)

# ✅ Step 7: Display Results in a Table
df = pd.DataFrame({
    "Week": list(range(1, len(weekly_orders) + 1)),
    "Actual Orders": weekly_orders,
    "Buffer Memory (Stores All)": buffer_memory_log,
    "Window Memory (Last 3 Turns)": window_memory_log,
    "Buffer Memory Prediction": buffer_predictions,
    "Window Memory Prediction": window_predictions
})


/tmp/ipykernel_1151/3228247777.py:20: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  window_memory = ConversationBufferWindowMemory(k=3, return_messages=True)  # Stores last 3 interactions


In [18]:
display(df)

,Week,Actual Orders,Buffer Memory (Stores All),Window Memory (Last 3 Turns),Buffer Memory Prediction,Window Memory Prediction
0,1,20,[content='Week 1: The previous orders were 20....,[content='Week 1: The previous orders were 20....,"To determine the next order quantity, it's imp...",To determine the order quantity in this scenar...
1,2,50,[content='Week 1: The previous orders were 20....,[content='Week 1: The previous orders were 20....,To determine the next order quantity based on ...,"When managing a supply chain, particularly for..."
2,3,10,[content='Week 1: The previous orders were 20....,[content='Week 1: The previous orders were 20....,To forecast the next order quantity for the be...,To predict the next order quantity in a beer d...
3,4,25,[content='Week 1: The previous orders were 20....,"[content='Week 2: The previous orders were 20,...","To determine the next order quantity, we can a...","To estimate the next order quantity, it's help..."
4,5,30,[content='Week 1: The previous orders were 20....,"[content='Week 3: The previous orders were 20,...","To determine the next order quantity, we can c...","To determine the next order quantity, we can a..."
5,6,30,[content='Week 1: The previous orders were 20....,"[content='Week 4: The previous orders were 20,...","To determine the next order quantity, we can a...",To estimate an appropriate order quantity for ...


In [19]:
# ✅ Save the table to an Excel file
df.to_excel("beer_game_memory_comparison.xlsx", index=False)

# ✅ Print confirmation message
print("Data saved to 'beer_game_memory_comparison.xlsx'")


Data saved to 'beer_game_memory_comparison.xlsx'


# 📌 **Assignment: AI Stock Market Trend Prediction with Memory**

## **Objective**
In this assignment, you will use AI to predict stock market trends based on historical stock prices. You will compare how different memory types affect AI's ability to track and predict future trends.

## **Tasks**
1. **Initialize memory types** (`ConversationBufferMemory` and `ConversationBufferWindowMemory`).
2. **Define the AI model** (GPT-4 or another suitable model).
3. **Complete the prompt template** to guide AI predictions.
4. **Process stock price data** and use memory to store past trends.
5. **Retrieve and analyze stored memory** after each step.
6. **Invoke the AI model correctly** to generate predictions.
7. **Save results to an Excel file** for analysis.

## **Expected Outcome**
You will observe how AI predictions change when it has full history vs. limited memory. This will help you understand the impact of memory in AI-based forecasting.

🚀 **Complete the placeholders and run the script to generate insights!** 🚀


In [20]:
# ==================================================
# ✋ Final Assignment: AI Stock Market Trend Prediction with Memory — COMPLETED
# ==================================================
# Compares ConversationBufferMemory (full history) vs.
# ConversationBufferWindowMemory (last 3 turns) for trend prediction.

import pandas as pd
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# ✅ Step 1: Initialize Memory Types
buffer_memory = ConversationBufferMemory(return_messages=True)      # full stock history
window_memory = ConversationBufferWindowMemory(k=3, return_messages=True)  # last 3 weeks

# ✅ Step 2: Initialize Chat Model
llm = ChatOpenAI(model=DEFAULT_CHAT_MODEL)

# ✅ Step 3: Define a Prompt Template
stock_prediction_template = PromptTemplate(
    input_variables=["context"],
    template="""
    You are an AI financial analyst predicting stock market trends.

    {context}

    Based on this stock price history, what will be the next trend (Up, Down, or Stable)?
    Answer in one short sentence and state Up / Down / Stable explicitly.
    """
)

# ✅ Step 4: Define Processing Pipelines (LCEL: prompt | model)
buffer_chain = stock_prediction_template | llm
window_chain = stock_prediction_template | llm

# ✅ Step 5: Define Stock Price Data
stock_prices = [120, 125, 110, 130, 128, 129]   # volatile first, then stabilising

buffer_memory_log, window_memory_log = [], []
buffer_predictions, window_predictions = [], []

# ✅ Step 6: Run the Prediction Simulation
for week in range(1, len(stock_prices) + 1):
    prev_prices = ", ".join(map(str, stock_prices[:week]))
    context = f"Week {week}: The previous stock prices were {prev_prices}."

    # Save the new turn into BOTH memory objects
    buffer_memory.save_context({"context": context}, {"response": ""})
    window_memory.save_context({"context": context}, {"response": ""})

    # Ask the model for a prediction using each chain
    buffer_prediction = buffer_chain.invoke({"context": context})
    window_prediction = window_chain.invoke({"context": context})

    # Snapshot what each memory currently holds
    buffer_memory_summary = buffer_memory.load_memory_variables({})["history"]
    window_memory_summary = window_memory.load_memory_variables({})["history"]

    buffer_memory_log.append(buffer_memory_summary)
    window_memory_log.append(window_memory_summary)
    buffer_predictions.append(buffer_prediction.content)
    window_predictions.append(window_prediction.content)

# ✅ Step 7: Save Results in an Excel File
df = pd.DataFrame({
    "Week": list(range(1, len(stock_prices) + 1)),
    "Stock Price": stock_prices,
    "Buffer Memory (Stores All)": buffer_memory_log,
    "Window Memory (Last 3 Turns)": window_memory_log,
    "Buffer Memory Prediction": buffer_predictions,
    "Window Memory Prediction": window_predictions,
})

df.to_excel("stock_market_memory_comparison.xlsx", index=False)
print("Assignment completed! Data saved to 'stock_market_memory_comparison.xlsx'")
display(df)


Assignment completed! Data saved to 'stock_market_memory_comparison.xlsx'


,Week,Stock Price,Buffer Memory (Stores All),Window Memory (Last 3 Turns),Buffer Memory Prediction,Window Memory Prediction
0,1,120,[content='Week 1: The previous stock prices we...,[content='Week 1: The previous stock prices we...,"Based on the given single data point, the tren...","Based on the single data point, the trend is S..."
1,2,125,[content='Week 1: The previous stock prices we...,[content='Week 1: The previous stock prices we...,"Based on the stock price history, the trend is...","Based on the previous stock prices, the next t..."
2,3,110,[content='Week 1: The previous stock prices we...,[content='Week 1: The previous stock prices we...,"Based on the previous stock prices, the trend ...","Based on the previous stock prices, the next t..."
3,4,130,[content='Week 1: The previous stock prices we...,[content='Week 2: The previous stock prices we...,Based on the recent volatility and fluctuation...,Week 4: The trend is likely to be Stable.
4,5,128,[content='Week 1: The previous stock prices we...,[content='Week 3: The previous stock prices we...,"Based on the stock price history, the next tre...","Based on the stock price history, the next tre..."
5,6,129,[content='Week 1: The previous stock prices we...,[content='Week 4: The previous stock prices we...,"Based on the stock price history, the next tre...","Based on the stock price history, the trend is..."
